In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
os.environ["PMIX_MCA_gds"]="hash"

# Import useful packages
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs
from qiskit_metal.toolbox_metal import math_and_overrides
from qiskit_metal.qlibrary.core import QComponent
from collections import OrderedDict
from qiskit_metal.qlibrary.qubits.transmon_cross import TransmonCross
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee


In [2]:
# Set up chip design as planar
design = designs.DesignPlanar({}, overwrite_enabled=True)

# Set up chip dimensions
design.chips.main.size.size_x = '4.6mm'
design.chips.main.size.size_y = '2.4mm'
design.chips.main.size.size_z = '-280um'
design.chips.main.size.center_x = '0mm'
design.chips.main.size.center_y = '-1mm'

# Resonator and feedline gap width (W) and center conductor width (S)
design.variables['cpw_width'] = '10 um' #S
design.variables['cpw_gap'] = '6 um' #W

# Create GUI
gui = MetalGUI(design)

In [3]:
# Lauchpad 1
x1 = '-2mm'
y1 = '0mm'
launch_options1 = dict(chip='main', pos_x=x1, pos_y=y1, orientation='360', lead_length='30um', pad_height='103um',
                      pad_width='103um', pad_gap='60um')
LP1 = LaunchpadWirebond(design, 'LP1', options = launch_options1)

# Launchpad 2
x2 = '2mm'
y1 = '0mm'
launch_options2 = dict(chip='main', pos_x=x2, pos_y=y1, orientation='180', lead_length='30um', pad_height='103um',
                      pad_width='103um', pad_gap='60um')
LP2 = LaunchpadWirebond(design, 'LP2', options = launch_options2)

# Using path finder to connect the two launchpads
TL = RoutePathfinder(design, 'TL', options = dict(chip='main', trace_width ='10um',
                                            trace_gap ='6um',
                                            fillet='90um',
                                            hfss_wire_bonds = True,
                                            lead=dict(end_straight='0.1mm'),
                                            pin_inputs=Dict(
                                                start_pin=Dict(
                                                    component='LP1',
                                                    pin='tie'),
                                                end_pin=Dict(
                                                    component='LP2',
                                                    pin='tie')
                                            )))

In [4]:
Q1 = TransmonCross(design, 'Q1', options = dict(pos_x = '0.6075mm', pos_y='-1.464',
                                                     connection_pads = dict(
                                                     bus_01 = dict(connector_location = '180',claw_length ='95um'),
                                                     readout = dict(connector_location = '0')),
                                                     fl_options = dict()))



In [5]:
#open to ground for resonator
otg1 = OpenToGround(design, 'otg1', options=dict(chip='main', pos_x='-0.2mm',  pos_y='-40um', orientation = 180))

# Use RouteMeander to fix the total length of the resonator
res1 = RouteMeander(design, 'resonator1',  Dict(
        trace_width ='10um',
        trace_gap ='6um',
        total_length='3.7mm',
        hfss_wire_bonds = False,
        fillet='99.9 um',
        lead = dict(start_straight='300um'),
        pin_inputs=Dict(
        start_pin=Dict(component= 'otg1', pin= 'open'),
        end_pin=Dict(component= 'Q1', pin= 'readout')), ))

# rebuild the GUI
gui.rebuild()
design.rebuild()

In [6]:
from SQDMetal.PALACE.Capacitance_Simulation import PALACE_Capacitance_Simulation

#Eigenmode Simulation Options
user_defined_options = {
                "mesh_refinement": 0,                             #refines mesh in PALACE - essetially divides every mesh element in half
                "dielectric_material": "silicon",                  #choose dielectric material - 'silicon' or 'sapphire'
                "solver_order": 2,                                 #increasing solver order increases accuracy of simulation, but significantly increases sim time
                "solns_to_save": 3,                                #number of solutions to save for E-field visualization - ordered by the entries in the computed capacitance matrix
                "solver_tol": 1.0e-8,                              #error residual tolerance for iterative solver
                "solver_maxits": 200,                              #number of solver iterations
                "fillet_resolution":12,                            #number of vertices per quarter turn on a filleted path
                "palace_dir":"/home/ruiz/repo/spack/opt/spack/linux-zen3/palace-develop-blpbtxe43ne2or3yo7gou6ycmdwdxuzl/bin/palace",#"PATH/TO/PALACE/BINARY",
                #e.g.: "palace_dir":"/home/prasanna/spack/opt/spack/linux-ubuntu24.04-zen2/gcc-13.2.0/palace-0.12.0-q65qvkwsa5zglixv3rmm424wqsu3mcpv/bin/palace"
                "num_cpus": 8
                }

#Create the Palace capacitance simulation
cap_sim = PALACE_Capacitance_Simulation(name = 'xmon_cap_sim',                                      #name of simulation
                                        metal_design = design,                                      #feed in qiskit metal design
                                        sim_parent_directory = "",                                  #choose directory where mesh file, config file and HPC batch file will be saved
                                        mode = 'simPC',                                             #choose simulation mode 'HPC' or 'simPC'
                                        meshing = 'GMSH',                                           #choose meshing 'GMSH' or 'COMSOL'
                                        user_options = user_defined_options,                        #provide options chosen above
                                        create_files = True)                                        #create mesh, config and HPC batch files

#Add metallic elements from layer 1 in design file
cap_sim.add_metallic(1)

#Add ground plane to simulations
cap_sim.add_ground_plane()

#Fine mesh the transmon cross qubit region
cap_sim.fine_mesh_components(['Q1'], min_size=12e-6, max_size=100e-6, taper_dist_min=10e-6, metals_only=False)

#Fine mesh resonator, launch pads and transmission line
cap_sim.fine_mesh_components(['resonator1', 'TL', 'LP1', 'LP2'], min_size=12e-6, max_size=120e-6, taper_dist_min=10e-6)

#This will prepare the mesh file and the config file
cap_sim.prepare_simulation()

Error   : Gmsh has not been initialized
Error   : Gmsh has not been initialized


In [7]:


#view the mesh
cap_sim.open_mesh_gmsh()



In [8]:

#Display the indices of the conductors to match with capacitance matrix - it will be automatically generated with the output anyway
cap_sim.display_conductor_indices()



<Figure size 640x480 with 1 Axes>

In [9]:
#run the simulation
capMat = cap_sim.run()



>> /usr/bin/mpirun -n 8 /home/ruiz/repo/spack/opt/spack/linux-zen3/palace-develop-blpbtxe43ne2or3yo7gou6ycmdwdxuzl/bin/palace-x86_64.bin xmon_cap_sim.json

_____________     _______
_____   __   \____ __   /____ ____________
____   /_/  /  __ ` /  /  __ ` /  ___/  _ \
___   _____/  /_/  /  /  /_/  /  /__/  ___/
  /__/     \___,__/__/\___,__/\_____\_____/


--> Warning!
Output folder is not empty; program will overwrite content! (outputFiles)
Git changeset ID: 3b8c020
Running with 8 MPI processes
Device configuration: cpu
Memory configuration: host-std
libCEED backend: /cpu/self/xsmm/blocked

Added 1338 boundary elements for material interfaces to the mesh

Characteristic length and time scales:
 Lc = 5.520e-03 m, tc = 1.841e-02 ns
Finished partitioning mesh into 8 subdomains

Mesh curvature order: 1
Mesh bounding box:
 (Xmin, Ymin, Zmin) = (-2.760e-03, -2.440e-03, -2.800e-04) m
 (Xmax, Ymax, Zmax) = (+2.760e-03, +4.400e-04, +2.800e-04) m

Parallel Mesh Stats:

                minimum  

In [10]:
# # View the raw capacitance matrix
# print(capMat)

# Or if you want it in fF (femtofarads), which is more readable for circuit QED:
print(capMat * 1e15)

[[ 6.22346096e+03 -8.91147482e+02 -6.53202866e+01 -1.03090685e+02
  -7.67298526e+02]
 [-8.91147482e+02  9.42087847e+02 -3.33765718e-08 -2.63729780e-07
  -1.62644393e+00]
 [-6.53202866e+01 -3.33765718e-08  6.99885561e+01 -2.46022824e+00
  -1.73640483e-03]
 [-1.03090685e+02 -2.63729780e-07 -2.46022824e+00  1.18429681e+02
  -9.72300939e-01]
 [-7.67298526e+02 -1.62644393e+00 -1.73640483e-03 -9.72300939e-01
   7.94160618e+02]]
